# Task 4 — YOLOv8 5-fold + masked pool (WBF 앙상블용 체크포인트 생성, Colab)

**목적**: task2-masked와 동일한 데이터(원본 train 232장 + 합성 pool2 + **masked pool**, 74종)로 **YOLOv8**을
5-fold 학습해, 추후 RF-DETR(task2-masked)와 fold-matched WBF 앙상블에 쓸 체크포인트 5개를 만든다.
**classification loss 가중치를 높여(cls gain 0.5→1.5)** 앙상블에서 분류 정확도에 기여하는 체크포인트를 목표로 한다.
이 노트북 자체의 산출물은 **체크포인트**이며, Kaggle 제출용 CSV는 만들지 않는다.

| 항목 | 내용 |
|---|---|
| 학습 데이터 | 원본 train 232장(corrections 반영) + 합성 pool2 + **masked pool** — **task2-masked와 완전히 동일** (`syn_00505/00102` 제외 포함) |
| 클래스 | 74종 (train 56종 → 라벨 1~56, test 전용 18종 → 라벨 57~74) |
| fold 분할 | **재계산하지 않음** — task2-masked에서 export한 `fold_split_masked.json`을 Drive에서 로드. 세션/계정/라이브러리 버전 차이로 인한 분할 불일치를 원천 차단 (fold-matched WBF의 전제) |
| 모델/학습 | **YOLOv8m** (YOLO11 대비 가벼움 — masked pool 추가로 데이터가 커져 시간 절약), 100 epochs, imgsz 960, patience 15 |
| loss | box=7.5, **cls=1.5(기본 0.5의 3배)**, dfl=1.5 — Ultralytics 내장 loss gain 인자로 조정 (**loss 코드 수정 없음**) |
| 역할 분담 | **이 노트북은 fold0 담당** (fold1~4는 Kaggle판 `task4_yolov8_5fold_masked_kaggle`) — test 추론/WBF는 별도 앙상블 노트북(`ensemble_wbf_inference_kaggle`)에서 수행 |

**masked pool 처리 (task2-masked와 동일한 유의사항)**
- 파일명이 train과 동일 체계(K코드+촬영조건)라 원본과 겹칠 수 있음 → **전체 파일을 `msk_` 접두어로 리네임**한
  사본을 스테이징 폴더에 만들어 병합 (이미지 캐시 덮어쓰기·corrections 오적용 방지)
- annotation은 **원본 category_id를 그대로 사용** (합성 pool의 라벨 네임스페이스와 다름 — name 매핑 불필요)
- 같은 구성(위도/촬영조건만 다른 이미지 + masked 버전)의 train/valid 누수 방지 그룹화는
  `fold_split_masked.json`에 이미 반영되어 있으며, 로드 후 그룹 누수 assert로 재검증한다
- **masked pool 경로는 자동 탐색하지 않음** → 4번 셀의 `MASKED_ROOT`에 실제 Drive 경로를 직접 기입할 것

**corrections 주의**
- task2-masked와 동일한 **구버전 스냅샷**을 하드코딩한다 (저장소 canonical과 일부 다름:
  fix_category `"3444"→3351`, `"791"→31863` 포함, add_boxes 1건 category 상이).
  앙상블 파트너인 task2-masked RF-DETR 체크포인트와 학습 라벨 조건을 완전히 일치시키기 위한 의도적 고정.

**실행 전제**
- Colab: **GPU 런타임** 필요 (Runtime → Change runtime type → GPU)
- Drive에 `fold_split_masked.json` 업로드 완료 상태여야 함 (4번 셀이 파일명으로 재귀 검색)
- 원칙: **저장소 공통 코드(`RF_DETR_split_ver`)는 수정하지 않는다.** 그대로 쓰기 어려운 부분(YOLO 변환,
  학습 루프 등)과 노트북 반복 코드는 **`yejin/pipeline` 모듈**로 분리했다 (셀 2에서 import).
- `batch`는 실제 배정받은 GPU(T4/L4/A100 등) 메모리에 따라 OOM 시 낮춰야 한다.


In [ ]:
# [1. 환경 준비] ultralytics(YOLOv8 학습/추론 + COCO->YOLO 변환 유틸 포함), ensemble-boxes(WBF)
!pip install -q "ultralytics>=8.3" ensemble-boxes


In [ ]:
# [2. 저장소 준비] 팀 저장소를 clone하고 공통 코드(RF_DETR_split_ver)와 yejin 모듈을 import 경로에 추가합니다.
#  ⚠ 저장소 공통 코드는 수정하지 않고 그대로 재사용합니다. YOLO 관련 로직은 yejin/pipeline.yolo 모듈에 있습니다.
import os, sys, re, glob, json, shutil, math
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from PIL import Image

REPO_URL = 'https://github.com/kim-tae-yoon-0718/ai12-team01.git'
REPO_DIR = '/content/ai12-team01'
REPO_BRANCH = 'main'   # ⚠ yejin 작업 브랜치가 main에 병합되기 전에 실행하려면 해당 브랜치명으로 변경
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
sys.path.insert(0, os.path.join(REPO_DIR, 'RF_DETR_split_ver'))
sys.path.insert(0, os.path.join(REPO_DIR, 'yejin'))

# ── 저장소 공통 코드(RF_DETR_split_ver)에서 그대로 재사용하는 함수 ─────────────
from colab_setup import mount_drive
from dataset import (
    find_data_root,       # Drive에서 폴더 이름으로 데이터 루트 자동 탐색
    check_data_paths,     # train/test 이미지·annotation 하위 경로 존재 점검
    load_raw_annotations, # 박스당 1개로 흩어진 원본 annotation을 파일명 기준 병합 로드
    apply_corrections,    # corrections 기반 bbox 수정
    cache_images,         # 이미지 로컬 캐시
    write_fold_dirs,      # fold별 {train,valid}에 COCO json + 이미지 배치
    save_label_map,       # label_map.json 저장
    compute_label_counts, # 클래스(라벨)별 박스 수 집계
)

# ── yejin/pipeline 공통 모듈 (노트북 반복 코드의 모듈화 - 각 함수 docstring 참고) ──
from pipeline import cloud, pools, folds, yolo
from pipeline.corrections import save_corrections_snapshot

print('저장소 clone 완료:', REPO_DIR)


In [ ]:
# [3. corrections 스냅샷] task2-masked와 완전히 동일한 yejin/pipeline 스냅샷을 사용합니다.
#  (fold-matched 앙상블 멤버 간 학습 라벨 조건 일치 목적 - pipeline/corrections.py 상단 주의 참고)
CORRECTIONS_PATH = save_corrections_snapshot('/content/corrections.json')


In [ ]:
# [4. Drive 마운트 + 경로 자동 탐색]
#  원본 train과 pool2는 팀 공유 Drive 폴더 하위에서 폴더 "이름"으로 찾습니다 (저장소 find_data_root).
#  masked pool은 Drive의 dataset 폴더에 zip(dataset_74_masked.zip)으로 있어 로컬에 풀어 사용합니다.
mount_drive()

SEARCH_ROOT = '/content/drive/MyDrive/(((((((내꺼)))))) 코드잇/스프린트 미션&피드백/초급프로젝트'

DATA_ROOT = find_data_root(
    candidates=[
        '/content/drive/MyDrive/1팀 공유 문서/ai12-level1-project/sprint_ai_project1_data',
        '/content/drive/MyDrive/1팀 공유 문서/박예진',
        '/content/drive/MyDrive/(((((((내꺼)))))) 코드잇/스프린트 미션&피드백/초급프로젝트',
    ],
    search_root=SEARCH_ROOT,
    target_name='sprint_ai_project1_data',
)
check_data_paths(DATA_ROOT)

POOL_ROOT = find_data_root(
    candidates=[
        '/content/drive/MyDrive/1팀 공유 문서/박예진/task2_synthesized',
        '/content/drive/MyDrive/(((((((내꺼)))))) 코드잇/스프린트 미션&피드백/초급프로젝트/dataset/task2_synthesized',
    ],
    search_root=SEARCH_ROOT,
    target_name='task2_synthesized',
)
POOL_IMG_DIR = os.path.join(POOL_ROOT, 'images')
POOL_ANN_PATH = (glob.glob(os.path.join(POOL_ROOT, 'annotations', '*.json')) or [None])[0]
assert POOL_ANN_PATH, 'pool2 annotation json을 못 찾음 - 폴더 구조 확인 필요 (images/, annotations/*.json)'

# ── masked pool: zip을 파일명으로 재귀 검색 -> 로컬 복사 후 압축 해제 (Drive 직접 해제보다 빠름) ──
MASKED_ZIP = cloud.find_file_under(SEARCH_ROOT, 'dataset_74_masked.zip')
MASKED_EXTRACT_DIR = '/content/masked_pool'
if not os.path.isdir(MASKED_EXTRACT_DIR):
    _local_zip = '/content/dataset_74_masked.zip'
    if not os.path.exists(_local_zip):
        shutil.copy(MASKED_ZIP, _local_zip)
    shutil.unpack_archive(_local_zip, MASKED_EXTRACT_DIR)
    os.remove(_local_zip)   # 압축 해제 후 zip 사본은 삭제해 디스크 확보
print('masked zip:', MASKED_ZIP, '\n-> 압축 해제:', MASKED_EXTRACT_DIR)

# zip 내부 폴더 구조가 어떻든 images/ 폴더와 annotations json을 이름으로 찾음
_img_hits = [p for p in glob.glob(os.path.join(MASKED_EXTRACT_DIR, '**', 'images'), recursive=True)
             if os.path.isdir(p) and '__MACOSX' not in p]
assert _img_hits, f'압축 해제 결과에서 images 폴더를 못 찾음: {MASKED_EXTRACT_DIR} 내부 구조 확인'
MASKED_IMG_DIR = _img_hits[0]
_ann_hits = [p for p in glob.glob(os.path.join(MASKED_EXTRACT_DIR, '**', 'annotations', '*.json'), recursive=True)
             if '__MACOSX' not in p]
assert _ann_hits, f'압축 해제 결과에서 annotations json을 못 찾음: {MASKED_EXTRACT_DIR} 내부 구조 확인'
MASKED_ANN_PATH = _ann_hits[0]

# ── 고정 fold 분할: task2-masked에서 export한 fold_split_masked.json을 Drive에서 파일명으로 검색 ──
FOLD_SPLIT_JSON = cloud.find_file_under(SEARCH_ROOT, 'fold_split_masked.json')

print('MASKED_IMG_DIR:', MASKED_IMG_DIR)
print('MASKED_ANN_PATH:', MASKED_ANN_PATH)
print('FOLD_SPLIT_JSON:', FOLD_SPLIT_JSON)


In [ ]:
# [5. 실험 설정]
#  task2-masked(RF-DETR medium, 100ep)와 동일한 데이터 조건에서 YOLOv8m을 학습합니다.
#  YOLO11 -> YOLOv8 변경: masked pool 추가로 데이터가 커져 학습 시간이 늘어난 상황에서 더 가벼운 아키텍처 선택.
#  imgsz=960 유지: 원본 976x1280의 알약 디테일 보존 (32의 배수).
#
#  loss gain — Ultralytics 내장 하이퍼파라미터로 loss 성분별 가중치를 조절합니다 (loss 코드 무수정).
#   total = box_gain*box_loss + cls_gain*cls_loss + dfl_gain*dfl_loss  (v8DetectionLoss)
#   기본값 box=7.5, cls=0.5, dfl=1.5 (box 위주) -> cls를 0.5에서 1.5(3배)로 올려 분류 오류에
#   더 큰 패널티를 부여. WBF 앙상블에서 classification 정확도에 기여하는 체크포인트가 목적.
#   ⚠ 학습 후 results.png에서 box_loss 수렴이 눈에 띄게 나빠졌는지 확인할 것 (나빠졌으면 cls 하향 재조정).
TASK_ID = 4
TAG = 'yolov8m_task4_syn74_masked'

config = {
    'data': {
        'n_splits': 5,
        'seed': 42,          # (참고) fold 분할은 fold_split_masked.json 로드로 고정 - seed는 분할에 관여하지 않음
        'dataset_dir': '/content/dataset',             # COCO 포맷 fold 디렉토리 (write_fold_dirs 산출물)
        'cache_dir': '/content/img_cache',
        'yolo_dataset_dir': '/content/yolo_dataset',   # YOLO 포맷(images/labels) fold 디렉토리
    },
    'model': {
        'variant': 'medium',   # nano | small | medium | large | xlarge (yolov8 계열)
        'tag': TAG,
    },
    'train': {
        'epochs': 100,
        'imgsz': 960,
        'batch': 8,             # Colab GPU 메모리에 따라 조정 (OOM 시 4로 축소)
        'patience': 15,         # val 개선이 15ep 없으면 조기 종료 (데이터 증가에 따른 시간 절약)
        'seed': 42,
        'box_gain': 7.5,        # 기본값 유지
        'cls_gain': 1.5,        # 기본 0.5의 3배 - classification loss 패널티 강화
        'dfl_gain': 1.5,        # 기본값 유지
    },
    'output': {
        'local_output_dir': '/content/outputs',
        # 체크포인트/학습곡선을 Drive에 영구 저장 - 세션이 끊겨도 재마운트만 하면 이어하기 가능
        'backup_dir': '/content/drive/MyDrive/(((((((내꺼)))))) 코드잇/스프린트 미션&피드백/초급프로젝트/0708 YOLO',
    },
}

for d in (config['data']['cache_dir'], config['data']['dataset_dir'],
          config['data']['yolo_dataset_dir'], config['output']['local_output_dir'],
          config['output']['backup_dir']):
    os.makedirs(d, exist_ok=True)

VIS_SCORE_THR = 0.3   # test 예측 시각화용 confidence threshold (QA 목적 - 제출용 아님)
WBF_IOU_THR = 0.55

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU(!) - 런타임 유형 확인 필요')
print('backup_dir(영구 저장 - Drive):', config['output']['backup_dir'])
print('loss gains:', {k: v for k, v in config['train'].items() if k.endswith('_gain')})


In [ ]:
# [6. 원본 train 로드 + annotation 수정] Task2와 완전히 동일한 절차입니다.
boxes_by_image, cats_by_image, img_meta, ids_by_image = load_raw_annotations(
    os.path.join(DATA_ROOT, 'train_annotations'))
print(f"수정 전: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

boxes_by_image, cats_by_image = apply_corrections(
    boxes_by_image, cats_by_image, ids_by_image, CORRECTIONS_PATH)
print(f"수정 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

train_cats = sorted({c for cs in cats_by_image.values() for c in cs})
print('train 클래스 수:', len(train_cats))


In [ ]:
# [7. pool2 병합] task2-masked와 완전히 동일한 로직입니다 (pipeline.pools).
#  ⚠ fold 구성(fold_split_masked.json)과 데이터 일치를 위해 병합 로직을 임의로 바꾸면 안 됩니다.
pool_boxes, pool_cats, pool_ids, pool_meta, pool_coco = pools.load_pool_annotations(POOL_ANN_PATH)
print(f"pool2: 이미지 {len(pool_meta)}장 / 박스 {sum(len(v) for v in pool_boxes.values())}개"
      f" / 클래스 {len({c for cs in pool_cats.values() for c in cs})}종")

pools.merge_pool(boxes_by_image, cats_by_image, ids_by_image, img_meta,
                 pool_boxes, pool_cats, pool_ids, pool_meta, pool_name='pool2')


In [ ]:
# [7-1. masked pool 병합] task2-masked와 완전히 동일한 로직입니다 (pipeline.pools.merge_masked_pool).
#  전체 파일을 'msk_' 접두어로 리네임한 사본을 스테이징 폴더에 만들어 병합합니다.
#  fold 그룹 키 계산 시 접두어를 벗기는 처리는 fold_split_masked.json에 이미 반영되어 있습니다.
MASKED_STAGE_DIR = '/content/masked_renamed'
cats_74 = {int(c['name']) for c in pool_coco['categories'] if c['id'] != 0}
n_masked, masked_coco = pools.merge_masked_pool(
    boxes_by_image, cats_by_image, ids_by_image, img_meta,
    MASKED_IMG_DIR, MASKED_ANN_PATH, MASKED_STAGE_DIR, cats_allowed=cats_74)


In [ ]:
# [7-2. 데이터 제외] task2-masked와 동일한 제외 목록을 반드시 그대로 적용합니다.
#  ⚠ fold_split_masked.json은 이 제외가 적용된 파일 집합 기준으로 만들어졌으므로,
#     목록이 다르면 fold 분할 로드 셀의 "분할-데이터 일치" assert에서 중단됩니다.
EXCLUDE_FILES = [
    'syn_00505.png',
    'syn_00102.png',
]
pools.apply_exclusions(EXCLUDE_FILES, boxes_by_image, cats_by_image, ids_by_image, img_meta)


In [ ]:
# [8. 74종 라벨 매핑] task2-masked와 동일한 체계 (pipeline.pools.build_cat2label_74).
#  fold 구성을 맞추려면 이 매핑도 완전히 동일해야 합니다 (층화 라벨 계산에 사용됨).
cat2label, label2cat, all_cats = pools.build_cat2label_74(pool_coco, train_cats, cats_by_image)


In [ ]:
# [9. 5-fold 분할 로드 + fold별 클래스 배분 점검] - 고정 분할 (재계산 없음)
#  StratifiedGroupKFold를 다시 돌리지 않고, task2-masked에서 export한 fold_split_masked.json을
#  로드합니다. 이유(재계산 시 세션/계정 간 sklearn·numpy 버전 차이로 분할이 달라질 위험 등)는
#  pipeline.folds docstring 참고. 분할-데이터 불일치는 load 단계 assert가 잡아줍니다.
file_names = sorted(boxes_by_image)
folds_split = folds.load_fold_split(FOLD_SPLIT_JSON, file_names, config['data']['n_splits'])
folds.assert_no_group_leak(folds_split, file_names)

fold_summary, valid_pivot = folds.summarize_fold_distribution(
    folds_split, file_names, cats_by_image, cat2label)
display(fold_summary)
folds.print_fold_warnings(fold_summary)
valid_pivot   # 라벨 x fold별 valid 박스 수 (클래스별 mAP 집계에서 재사용)


In [ ]:
# 이 셀 바로 위에 추가해서 확인
print("=== category mapping 확인 ===")
print(f"총 클래스 수: {len(cat2label)}")
print(f"\n{'category_id':>12} {'dl_idx':>10}")
print("-" * 25)
for name, cat_id in sorted(cat2label.items(), key=lambda x: x[1]):
    print(f"  {cat_id:>10} {name:>10}")

In [ ]:
# [10. COCO 포맷 fold 디렉토리 생성] task2-masked와 동일한 저장소 함수 조합입니다.
#  이 산출물(fold{i}/{train,valid}/_annotations.coco.json + 이미지)을 다음 셀에서 YOLO 포맷으로 변환합니다.
cache_images(os.path.join(DATA_ROOT, 'train_images'), config['data']['cache_dir'])
cache_images(POOL_IMG_DIR, config['data']['cache_dir'])
cache_images(MASKED_STAGE_DIR, config['data']['cache_dir'])   # masked 리네임 사본(msk_*.png)도 추가

write_fold_dirs(folds_split, file_names, boxes_by_image, cats_by_image, img_meta,
                cat2label, all_cats, config['data']['cache_dir'], config['data']['dataset_dir'])

save_label_map(cat2label, label2cat, config['data']['dataset_dir'])
print('COCO 포맷 fold 디렉토리 생성 완료:', config['data']['dataset_dir'])

# fold 디렉토리 복사가 끝나면 캐시는 더 필요 없으므로 삭제해 디스크를 확보합니다
shutil.rmtree(config['data']['cache_dir'], ignore_errors=True)
print('이미지 캐시 삭제 완료 (fold 디렉토리에 이미 복사됨)')


## 11. YOLO 포맷 변환

Ultralytics는 객체 검출 학습에 COCO json을 직접 쓰지 않고 `images/` + `labels/`(YOLO txt) 구조와
`data.yaml`을 요구한다 (확인 결과: 세그멘테이션/포즈가 아닌 detection 학습에 COCO json을 그대로 읽는
네이티브 경로는 없음). 다만 공식 변환 유틸 `ultralytics.data.converter.convert_coco()`가 이 변환을
대신해주므로 직접 파싱하지 않고 그대로 사용한다.

- `cls91to80=False`로 호출하면 `class_index = category_id - 1`을 그대로 쓴다 (COCO 80/91클래스
  리매핑을 켜지 않음 — 우리 데이터는 원래 COCO 80종이 아니므로 반드시 꺼야 함).
- 저장소 `build_coco()`가 넣는 더미 배경 카테고리(`id=0`, name='pill')에는 박스가 하나도 없으므로,
  별도로 제거하지 않아도 실제 74종이 정확히 YOLO class index `0~73`으로 매핑된다.
- `convert_coco()`는 라벨 txt만 생성하고 이미지는 복사하지 않으므로, 이미지는 심볼릭 링크로 연결해
  디스크 중복을 피한다.
- 변환된 txt 파일명은 COCO json의 `file_name`과 동일한 stem을 쓰므로, `images/`와 `labels/`를
  같은 파일명 규칙으로 나란히 두면 Ultralytics가 자동으로 짝을 찾는다.


In [ ]:
# [11. YOLO 포맷 변환 실행] COCO fold -> YOLO(images/labels+data.yaml) 변환.
#  convert_coco() 활용 방식과 클래스 인덱스 규약은 pipeline.yolo.build_yolo_fold docstring 참고.
fold_yaml_paths = [
    yolo.build_yolo_fold(fi, config['data']['dataset_dir'],
                         config['data']['yolo_dataset_dir'], label2cat)
    for fi in range(config['data']['n_splits'])
]
print('data.yaml 경로:')
for p in fold_yaml_paths:
    print(' ', p)


## 12. 지정 fold 학습 실행 (역할 분담: 이 노트북 = fold0)

fold1~4는 Kaggle판(`task4_yolov8_5fold_masked_kaggle`)이 담당합니다. 체크포인트 파일명 체계가
동일하므로(`{tag}_fold{i}_best.pt`) 나중에 한 폴더에 모으면 앙상블 노트북에서 그대로 사용합니다.

- `pipeline.yolo.train_fold_yolo()`는 `backup_dir`(Drive)에 해당 fold의 best.pt가 이미 있으면
  자동으로 건너뜁니다 → 세션이 끊겨도 재마운트 후 재실행하면 이어하기가 됩니다.
- fold 구성은 `fold_split_masked.json` 로드로 고정되어 세션/계정이 달라져도 동일하게 유지됩니다
  (데이터 소스나 제외 목록이 바뀌면 로드 셀 assert에서 중단되므로 조용히 어긋날 수 없음).


In [ ]:
# [12. 지정 fold 학습 + 리포팅 실행] (pipeline.yolo.run_folds_yolo)
FOLDS_THIS_SESSION = [0]   # 이 노트북(Colab)은 fold0 담당 - fold1~4는 Kaggle판에서

run_result = yolo.run_folds_yolo(config, fold_yaml_paths, fold_indices=FOLDS_THIS_SESSION)


In [ ]:
# [13. fold 결과 요약] (pipeline.yolo.summarize_* - 저장소 summarize_*의 YOLO 버전)
#  여러 fold를 완료한 뒤 실행하면 fold 평균±표준편차와 클래스별 mAP 집계표가 나옵니다.
kfold_summary = yolo.summarize_kfold_results_yolo(run_result['fold_metrics'], config['model']['tag'])

label_counts = compute_label_counts(config['data']['dataset_dir'])
per_class_df = yolo.summarize_per_class_yolo(run_result['fold_metrics'], label2cat,
                                             label_counts, valid_pivot)
per_class_df   # mean_AP 내림차순 - RF-DETR(task2-masked) 결과와 클래스별로 대조해볼 수 있음


## 산출물

- `{backup_dir}/{tag}_fold0_best.pt` — fold0 체크포인트 (Drive에 영구 저장, cls gain 1.5로 분류 강화 학습)
- `{tag}_fold0_results.csv` / `_results.png` — 학습 곡선 (Ultralytics 자동 생성)
- fold1~4는 Kaggle판이 생산하며, **test 추론/WBF/제출 CSV는 전 체크포인트를 모아
  `ensemble_wbf_inference_kaggle` 노트북에서 일괄 수행**합니다.
